In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
import numpy as np
import time
import os
from pathlib import Path
import json

In [2]:
df = pd.read_csv('anime_df.csv')
df.columns.values[0] = 'usernames'

In [3]:
df = df.set_index(df.columns[0])
df = df.astype(float)

import numpy as np

def get_recommendations(target_idx, df_features, df_orig, n=5):
    # 1. Convert to values (numpy) to avoid index errors
    matrix_norm = normalize(df_features.values)

# 2. Pobieramy wektor użytkownika z już znormalizowanej macierzy
    target_vec = matrix_norm[target_idx]
    
    # 3. The "Magic" Speed Line: Dot Product
    # This calculates similarity for ALL rows at once in one CPU cycle
    # (Assuming you normalized df_features once during init)
    scores = np.dot(matrix_norm, target_vec)
    
    # 4. Get top N (plus 1 because the most similar is the anime itself)
    # argpartition is faster than sorting the whole list
    top_indices = np.argpartition(scores, -(n+1))[-(n+1):]
    
    # 5. Map back to original IDs/Names using df_orig
    results = {}
    # top_indices to numery wierszy/kolumn, które są najbardziej podobne
    for i in top_indices:
        if i == target_idx: continue
        
        # Pobieramy nazwę prosto z listy kolumn (bo tam są Twoje tytuły)
        anime_full_name = df_orig.columns[i]
        
        # Dodajemy do wyników
        results[anime_full_name] = float(scores[i])

        
    return results

In [4]:
t0 = time.time()

In [5]:
conf_path = Path("../tauri.conf.json")

with open(conf_path, 'r') as f:
    config = json.load(f)

In [6]:
app_data_path = Path(os.getenv('APPDATA') or Path.home() / ".local/share")

# Path objects handle joining naturally with the / operator
watchlist_path = app_data_path / config['identifier'] / 'watchlist.json'

print(f"Watchlist path: {watchlist_path}")
print(f"DEBUG: Próbuję czytać z: {os.path.abspath(watchlist_path)}")

Watchlist path: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json
DEBUG: Próbuję czytać z: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json


In [7]:
user_data = pd.read_json(watchlist_path)

# Konwersja na listę słowników
user_data = user_data.to_dict(orient='records')

In [8]:
user_ratings = {
    #f"{entry['mal_id']}": f"{entry['score']}"
    f"{entry['mal_id']}_{entry['title']}": 7.0
    for entry in user_data
}

# print(user_ratings)

In [9]:
new_user_name = "Current_User"

new_user_row = pd.Series(0, index=df.columns, name=new_user_name)

for anime, rating in user_ratings.items():
    if anime in new_user_row.index:
        new_user_row[anime] = rating


df_extended = pd.concat([df, new_user_row.to_frame().T])
df_normalized = df_extended.apply(lambda row: row - row[row != 0].mean() if (row != 0).any() else row, axis=1)

usernames = df_extended.index

# To liczy tylko 1 x 5000 porównań (dla Current_User)
user_sim_vector = cosine_similarity(df_normalized.loc[["Current_User"]], df_normalized)

# Tworzymy DataFrame o wymiarach (1, liczba_użytkowników)
user_sim_df = pd.DataFrame(user_sim_vector, index=["Current_User"], columns=df_extended.index)

In [10]:
recommendations = get_recommendations(-1, df_extended, df_normalized, n=5)


for anime_key, score in recommendations.items():
    id_ref, name = anime_key.split('_', 1)
    print(f"Recommend: {name} (ID: {id_ref}) with predicted score: {score:.2f}")

Recommend: Azumanga Daiou The Animation (ID: 66) with predicted score: 0.27
Recommend: Blassreiter (ID: 3407) with predicted score: 0.28
Recommend: Bubblegum Crisis Tokyo 2040 (ID: 568) with predicted score: 0.28
Recommend: Armitage III: Dual-Matrix (ID: 492) with predicted score: 0.29
Recommend: Berserk: Ougon Jidai-hen I - Haou no Tamago (ID: 10218) with predicted score: 0.33
Recommend: Platonic Chain (ID: 611) with predicted score: 1.00


In [11]:
print(time.time() - t0)

1.2006652355194092
